## Shared infrastructure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import csv
import warnings
import pathlib

OUTPUT_DIR = pathlib.Path(r'C:\Dev\Master_thesis_final\Independent_Output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# silence LightGBM feature-name warning
warnings.filterwarnings(
    'ignore',
    message='X does not have valid feature names',
    category=UserWarning,
)
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not available.')

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print('LightGBM not available.')

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

np.random.seed(42)
_current_scenario_seed = None  # path cache disabled here (always None)


def poisson(lamb, T):
    t = np.cumsum(np.random.exponential(1 / lamb, size=int(lamb * T * 2)))
    return t[t <= T]

def gbm_paths(S0, r, sigma, T, lamb, M):
    paths, times = [], []
    for _ in range(M):
        t = poisson(lamb, T)
        if len(t) == 0:
            t = np.array([0, T])
        else:
            t = np.insert(t, 0, 0)
            if t[-1] < T:
                t = np.append(t, T)
        B = np.zeros(len(t))
        for i in range(1, len(t)):
            dt = t[i] - t[i - 1]
            B[i] = B[i - 1] + np.random.normal(0, np.sqrt(dt))
        paths.append(S0 * np.exp((r - 0.5 * sigma**2) * t + sigma * B))
        times.append(t)
    return paths, times

# path cache so paired IS/OOS calls reuse the same paths within a scenario

from collections import OrderedDict as _OrderedDict
_PATH_CACHE = _OrderedDict()
_PATH_CACHE_MAX = 4

_raw_gbm_paths = gbm_paths

def gbm_paths(S0, r, sigma, T, lamb, M):
    if _current_scenario_seed is None:
        return _raw_gbm_paths(S0, r, sigma, T, lamb, M)
    key = (float(S0), float(r), float(sigma), float(T),
           int(lamb), int(M), int(_current_scenario_seed))
    if key in _PATH_CACHE:
        _PATH_CACHE.move_to_end(key)
        return _PATH_CACHE[key]
    result = _raw_gbm_paths(S0, r, sigma, T, lamb, M)
    _PATH_CACHE[key] = result
    if len(_PATH_CACHE) > _PATH_CACHE_MAX:
        _PATH_CACHE.popitem(last=False)
    return result


def payoff(S, strike):
    return np.maximum(strike - S, 0)

def create_features(S, t, T, r, sigma, strike):
    tte = T - t
    return np.array([
        S, tte, S**2,
        np.log(S) if S > 0 else 0.0,
        np.sqrt(S) if S > 0 else 0.0,
        S / strike,
        sigma * np.sqrt(tte),
        r * tte,
    ])


class MLModelManager:
    def __init__(self, model_type='random_forest'):
        self.model_type = model_type
        self.model = self._create_model()

    def _create_model(self):
        mt = self.model_type
        if mt == 'xgboost' and XGBOOST_AVAILABLE:
            return xgb.XGBRegressor(
                n_estimators=30, max_depth=3, learning_rate=0.1, random_state=42)
        if mt == 'lightgbm' and LIGHTGBM_AVAILABLE:
            return lgb.LGBMRegressor(
                n_estimators=30, max_depth=3, learning_rate=0.1,
                random_state=42, verbose=-1)
        if mt == 'random_forest':
            return RandomForestRegressor(
            n_estimators=30, max_depth=3, max_features=3, min_samples_leaf=5,
            random_state=42)
        if mt == 'knn':
            return Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsRegressor(n_neighbors=200)),
    ])
        if mt == 'decision_tree':
            return DecisionTreeRegressor(max_depth=3, random_state=42)
        if mt in ('logistic_regression', 'ridge'):
            return Ridge(alpha=1.0)
        return KNeighborsRegressor(n_neighbors=200)

    def fit(self, X, y):
        if len(X) > 0:
            self.model.fit(X, y)

    def predict(self, X):
        return self.model.predict(X) if len(X) > 0 else np.array([])

# classical LSMC with basis [1, S, S^2]

def _classical_lsmc(S0, strike, r, sigma, T, M, lamb, regressor='lstsq'):
    all_paths, all_time_grids = gbm_paths(S0, r, sigma, T, lamb, M)
    cash_flows = [float(payoff(p[-1], strike)) for p in all_paths]
    cash_times = [float(t_grid[-1]) for t_grid in all_time_grids]
    max_steps = max(len(p) for p in all_paths)
    fit_time = 0.0

    for step in range(max_steps - 2, 0, -1):
        path_indices, X_list, Y_list = [], [], []

        for idx, (path, t_grid) in enumerate(zip(all_paths, all_time_grids)):
            if step >= len(path):
                continue
            current_price = path[step]
            if current_price >= strike:
                continue
            current_time = float(t_grid[step])
            path_indices.append(idx)
            X_list.append(current_price)
            Y_list.append(cash_flows[idx] * np.exp(-r * (cash_times[idx] - current_time)))

        if len(path_indices) < 3:
            continue

        X = np.array(X_list)
        Y = np.array(Y_list)
        A = np.vstack([X, X**2]).T

        try:
            t0 = time.time()
            if regressor == 'ridge':
                m = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
                m.fit(A, Y)
                continuation = m.predict(A)
            else:
                A_ls = np.column_stack([np.ones_like(X), A])
                coeff = np.linalg.lstsq(A_ls, Y, rcond=None)[0]
                continuation = A_ls @ coeff
            fit_time += time.time() - t0
        except Exception:
            continue

        for i, idx in enumerate(path_indices):
            ex_val = float(payoff(all_paths[idx][step], strike))
            if ex_val > continuation[i]:
                cash_flows[idx] = ex_val
                cash_times[idx] = float(all_time_grids[idx][step])

    discounted = np.array([cf * np.exp(-r * ct)
                           for cf, ct in zip(cash_flows, cash_times)])
    continuation_at_zero = float(discounted.mean())
    ex_val_at_zero = float(payoff(S0, strike))
    price = ex_val_at_zero if ex_val_at_zero > continuation_at_zero else continuation_at_zero
    stderr = float(discounted.std() / np.sqrt(len(discounted)))
    return price, fit_time, stderr

def LSMC_poly(S0, strike, r, sigma, T, M, lamb):
    val, _, stderr = _classical_lsmc(S0, strike, r, sigma, T, M, lamb, regressor='lstsq')
    return val, stderr

# backward iteration

def _lsmc_backward(all_paths, all_time_grids, cash_flows, cash_times, max_steps,
                   strike, r, sigma, T, model_type, collect_step=None):
    total_ml_time = 0.0
    plot_data = None

    for step in range(max_steps - 2, 0, -1):
        path_indices, X_list, Y_list = [], [], []

        for idx, (path, t_grid) in enumerate(zip(all_paths, all_time_grids)):
            if step >= len(path):
                continue
            current_price = path[step]
            exercise_value = payoff(current_price, strike)
            if exercise_value <= 1e-6:
                continue
            current_time = float(t_grid[step])
            path_indices.append(idx)
            X_list.append(create_features(current_price, current_time, T, r, sigma, strike))
            Y_list.append(cash_flows[idx] * np.exp(-r * (cash_times[idx] - current_time)))

        if len(path_indices) < 10:
            continue

        X, Y = np.array(X_list), np.array(Y_list)

        if collect_step is not None and step == collect_step:
            plot_data = {
                'stock_prices': np.array([all_paths[i][step] for i in path_indices]),
                'continuation_values': Y,
            }

        if np.std(Y) <= 1e-6:
            continue

        try:
            t0 = time.time()
            mdl = MLModelManager(model_type)
            mdl.fit(X, Y)
            pred = mdl.predict(X)
            total_ml_time += time.time() - t0
        except Exception:
            continue

        for i, idx in enumerate(path_indices):
            ex_val = float(payoff(all_paths[idx][step], strike))
            if ex_val > pred[i]:
                cash_flows[idx] = ex_val
                cash_times[idx] = float(all_time_grids[idx][step])

    return cash_flows, cash_times, total_ml_time, plot_data


def LSMC_ML(S0, strike, r, sigma, T, M, lamb, model_type='random_forest'):
    # ridge uses classical 1/S/S^2 path
    if model_type in ('logistic_regression', 'ridge'):
        return _classical_lsmc(S0, strike, r, sigma, T, M, lamb, regressor='ridge')

    all_paths, all_time_grids = gbm_paths(S0, r, sigma, T, lamb, M)
    cash_flows = [float(payoff(p[-1], strike)) for p in all_paths]
    cash_times = [float(t_grid[-1]) for t_grid in all_time_grids]
    max_steps = max(len(p) for p in all_paths)
    cash_flows, cash_times, ml_time, _ = _lsmc_backward(
    all_paths, all_time_grids, cash_flows, cash_times, max_steps,
    strike, r, sigma, T, model_type)
    discounted = np.array([cf * np.exp(-r * ct)
                           for cf, ct in zip(cash_flows, cash_times)])
    continuation_at_zero = float(discounted.mean())
    ex_val_at_zero = float(payoff(S0, strike))
    price = ex_val_at_zero if ex_val_at_zero > continuation_at_zero else continuation_at_zero
    stderr = float(discounted.std() / np.sqrt(len(discounted)))
    return price, ml_time, stderr

def LSMC_with_plot_data(S0, strike, r, sigma, T, M, lamb, model_type):
    all_paths, all_time_grids = gbm_paths(S0, r, sigma, T, lamb, M)
    cash_flows = [float(payoff(p[-1], strike)) for p in all_paths]
    cash_times = [float(t_grid[-1]) for t_grid in all_time_grids]
    max_steps = max(len(p) for p in all_paths)
    target_step = max(1, max_steps // 2)
    _, _, _, plot_data = _lsmc_backward(
    all_paths, all_time_grids, cash_flows, cash_times, max_steps,
    strike, r, sigma, T, model_type, collect_step=target_step)
    return plot_data


# K=5 folds, every path's exercise decision uses a regressor trained without it
OOS_N_FOLDS = 5

def _classical_lsmc_oos(S0, strike, r, sigma, T, M, lamb, n_folds=OOS_N_FOLDS):
    all_paths, all_time_grids = gbm_paths(S0, r, sigma, T, lamb, M)
    fold_ids = np.arange(M) % n_folds
    cash_flows = [float(payoff(p[-1], strike)) for p in all_paths]
    cash_times = [float(t_grid[-1]) for t_grid in all_time_grids]
    max_steps = max(len(p) for p in all_paths)
    fit_time = 0.0

    for step in range(max_steps - 2, 0, -1):
        idx_at_step, X_list, Y_list = [], [], []
        for idx, (path, t_grid) in enumerate(zip(all_paths, all_time_grids)):
            if step >= len(path):
                continue
            current_price = path[step]
            if current_price >= strike:
                continue
            current_time = float(t_grid[step])
            idx_at_step.append(idx)
            X_list.append(current_price)
            Y_list.append(cash_flows[idx] * np.exp(-r * (cash_times[idx] - current_time)))

        if len(idx_at_step) < n_folds * 3:
            continue

        X = np.array(X_list)
        Y = np.array(Y_list)
        A = np.vstack([X, X**2]).T
        step_fold = np.array([fold_ids[i] for i in idx_at_step])

        continuation = np.empty(len(idx_at_step))
        any_fit = False
        for k in range(n_folds):
            train_mask = step_fold != k
            eval_mask  = step_fold == k
            if train_mask.sum() < 3 or eval_mask.sum() == 0:
                continuation[eval_mask] = -np.inf
                continue
            try:
                t0 = time.time()
                m = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
                m.fit(A[train_mask], Y[train_mask])
                continuation[eval_mask] = m.predict(A[eval_mask])
                fit_time += time.time() - t0
                any_fit = True
            except Exception:
                continuation[eval_mask] = -np.inf

        if not any_fit:
            continue

        for i, idx in enumerate(idx_at_step):
            ex_val = float(payoff(all_paths[idx][step], strike))
            if ex_val > continuation[i]:
                cash_flows[idx] = ex_val
                cash_times[idx] = float(all_time_grids[idx][step])

    discounted = np.array([cf * np.exp(-r * ct)
                           for cf, ct in zip(cash_flows, cash_times)])
    continuation_at_zero = float(discounted.mean())
    ex_val_at_zero = float(payoff(S0, strike))
    price = ex_val_at_zero if ex_val_at_zero > continuation_at_zero else continuation_at_zero
    stderr = float(discounted.std() / np.sqrt(len(discounted)))
    return price, fit_time, stderr


def LSMC_ML_oos(S0, strike, r, sigma, T, M, lamb, model_type='random_forest',
                n_folds=OOS_N_FOLDS):
    if model_type in ('logistic_regression', 'ridge'):
        return _classical_lsmc_oos(S0, strike, r, sigma, T, M, lamb,
                                    n_folds=n_folds)

    all_paths, all_time_grids = gbm_paths(S0, r, sigma, T, lamb, M)
    fold_ids = np.arange(M) % n_folds

    cash_flows = [float(payoff(p[-1], strike)) for p in all_paths]
    cash_times = [float(t_grid[-1]) for t_grid in all_time_grids]
    max_steps = max(len(p) for p in all_paths)
    total_ml_time = 0.0

    for step in range(max_steps - 2, 0, -1):
        all_X, all_Y, idx_at_step = [], [], []
        for idx, (path, t_grid) in enumerate(zip(all_paths, all_time_grids)):
            if step >= len(path):
                continue
            current_price = path[step]
            if payoff(current_price, strike) <= 1e-6:
                continue
            current_time = float(t_grid[step])
            all_X.append(create_features(current_price, current_time, T, r, sigma, strike))
            all_Y.append(cash_flows[idx] * np.exp(-r * (cash_times[idx] - current_time)))
            idx_at_step.append(idx)

        if len(idx_at_step) < n_folds * 5:
            continue

        all_X = np.array(all_X)
        all_Y = np.array(all_Y)
        step_fold = np.array([fold_ids[i] for i in idx_at_step])

        if np.std(all_Y) <= 1e-6:
            continue

        pred = np.empty(len(idx_at_step))
        any_fit = False
        for k in range(n_folds):
            train_mask = step_fold != k
            eval_mask  = step_fold == k
            train_X = all_X[train_mask]
            train_Y = all_Y[train_mask]
            if len(train_X) < 10 or np.std(train_Y) <= 1e-6 or eval_mask.sum() == 0:
                pred[eval_mask] = -np.inf
                continue
            try:
                t0 = time.time()
                mdl = MLModelManager(model_type)
                mdl.fit(train_X, train_Y)
                pred[eval_mask] = mdl.predict(all_X[eval_mask])
                total_ml_time += time.time() - t0
                any_fit = True
            except Exception:
                pred[eval_mask] = -np.inf

        if not any_fit:
            continue

        for i, idx in enumerate(idx_at_step):
            ex_val = float(payoff(all_paths[idx][step], strike))
            if ex_val > pred[i]:
                cash_flows[idx] = ex_val
                cash_times[idx] = float(all_time_grids[idx][step])

    discounted = np.array([cf * np.exp(-r * ct)
                           for cf, ct in zip(cash_flows, cash_times)])
    continuation_at_zero = float(discounted.mean())
    ex_val_at_zero = float(payoff(S0, strike))
    price = ex_val_at_zero if ex_val_at_zero > continuation_at_zero else continuation_at_zero
    stderr = float(discounted.std() / np.sqrt(len(discounted)))
    return price, total_ml_time, stderr

# reference price: cubic basis [1,S,S^2,S^3] with large M_ref, bias target

def LSMC_reference(S0, strike, r, sigma, T, M, lamb):
    all_paths, all_time_grids = gbm_paths(S0, r, sigma, T, lamb, M)
    cash_flows = [float(payoff(p[-1], strike)) for p in all_paths]
    cash_times = [float(t_grid[-1]) for t_grid in all_time_grids]
    max_steps = max(len(p) for p in all_paths)
    for step in range(max_steps - 2, 0, -1):
        path_indices, X_list, Y_list = [], [], []
        for idx, (path, t_grid) in enumerate(zip(all_paths, all_time_grids)):
            if step >= len(path):
                continue
            current_price = path[step]
            if current_price >= strike:
                continue
            current_time = float(t_grid[step])
            path_indices.append(idx)
            X_list.append(current_price)
            Y_list.append(cash_flows[idx] * np.exp(-r * (cash_times[idx] - current_time)))
        if len(path_indices) < 4:
            continue
        X = np.array(X_list); Y = np.array(Y_list)
        A = np.vstack([np.ones_like(X), X, X**2, X**3]).T
        try:
            coeff = np.linalg.lstsq(A, Y, rcond=None)[0]
            continuation = A @ coeff
        except Exception:
            continue
        for i, idx in enumerate(path_indices):
            ex_val = float(payoff(all_paths[idx][step], strike))
            if ex_val > continuation[i]:
                cash_flows[idx] = ex_val
                cash_times[idx] = float(all_time_grids[idx][step])
    discounted = np.array([cf * np.exp(-r * ct) for cf, ct in zip(cash_flows, cash_times)])
    continuation_at_zero = float(discounted.mean())
    ex_val_at_zero = float(payoff(S0, strike))
    price = ex_val_at_zero if ex_val_at_zero > continuation_at_zero else continuation_at_zero
    return price, float(discounted.std() / np.sqrt(len(discounted)))

# K-fold cubic reference independent of any path's cash flow

def LSMC_reference_oos(S0, strike, r, sigma, T, M, lamb, n_folds=OOS_N_FOLDS):
    all_paths, all_time_grids = gbm_paths(S0, r, sigma, T, lamb, M)
    fold_ids = np.arange(M) % n_folds
    cash_flows = [float(payoff(p[-1], strike)) for p in all_paths]
    cash_times = [float(t_grid[-1]) for t_grid in all_time_grids]
    max_steps = max(len(p) for p in all_paths)

    for step in range(max_steps - 2, 0, -1):
        idx_at_step, X_list, Y_list = [], [], []
        for idx, (path, t_grid) in enumerate(zip(all_paths, all_time_grids)):
            if step >= len(path):
                continue
            current_price = path[step]
            if current_price >= strike:
                continue
            current_time = float(t_grid[step])
            idx_at_step.append(idx)
            X_list.append(current_price)
            Y_list.append(cash_flows[idx] * np.exp(-r * (cash_times[idx] - current_time)))

        if len(idx_at_step) < n_folds * 4:
            continue

        X = np.array(X_list)
        Y = np.array(Y_list)
        A = np.vstack([np.ones_like(X), X, X**2, X**3]).T
        step_fold = np.array([fold_ids[i] for i in idx_at_step])

        continuation = np.empty(len(idx_at_step))
        any_fit = False
        for k in range(n_folds):
            train_mask = step_fold != k
            eval_mask  = step_fold == k
            if train_mask.sum() < 4 or eval_mask.sum() == 0:
                continuation[eval_mask] = -np.inf
                continue
            try:
                coeff = np.linalg.lstsq(A[train_mask], Y[train_mask], rcond=None)[0]
                continuation[eval_mask] = A[eval_mask] @ coeff
                any_fit = True
            except Exception:
                continuation[eval_mask] = -np.inf

        if not any_fit:
            continue

        for i, idx in enumerate(idx_at_step):
            ex_val = float(payoff(all_paths[idx][step], strike))
            if ex_val > continuation[i]:
                cash_flows[idx] = ex_val
                cash_times[idx] = float(all_time_grids[idx][step])

    discounted = np.array([cf * np.exp(-r * ct)
                           for cf, ct in zip(cash_flows, cash_times)])
    continuation_at_zero = float(discounted.mean())
    ex_val_at_zero = float(payoff(S0, strike))
    price = ex_val_at_zero if ex_val_at_zero > continuation_at_zero else continuation_at_zero
    return price, float(discounted.std() / np.sqrt(len(discounted)))

# cache the K-fold cubic reference per scenario

_REF_CACHE = {}

_raw_LSMC_reference_oos = LSMC_reference_oos

def LSMC_reference_oos(S0, strike, r, sigma, T, M, lamb, n_folds=OOS_N_FOLDS):
    key = (float(S0), float(strike), float(r), float(sigma),
           float(T), int(M), int(lamb), int(n_folds))
    if key in _REF_CACHE:
        return _REF_CACHE[key]
    result = _raw_LSMC_reference_oos(S0, strike, r, sigma, T, M, lamb, n_folds)
    _REF_CACHE[key] = result
    return result


## IS Model comparison: plot

In [ ]:
LAMBDAS = [10, 25, 50, 100]
T, M, r, sigma, strike = 1.0, 10000, 0.05, 0.2, 90
test_s0 = 100

stock_prices_grid = np.linspace(70, 130, 30)

available_models = ['knn', 'decision_tree', 'random_forest', 'logistic_regression']
if XGBOOST_AVAILABLE:
    available_models.insert(2, 'xgboost')
if LIGHTGBM_AVAILABLE:
    available_models.insert(3 if XGBOOST_AVAILABLE else 2, 'lightgbm')

payoffs = [payoff(S, strike) for S in stock_prices_grid]
model_labels = {m: m.upper() for m in available_models}
model_labels['lsmc poly'] = 'LSMC Poly (1+S+S²)'

colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'teal']
styles = ['-', '--', '-.', ':', '-', '--', '-.']

for lamb in LAMBDAS:
    print('\n' + '-' * 40)
    print(f'Lambda = {lamb}')
    print('-' * 40)

    results = {}
    for model in available_models:
        print(f'  Computing IS  for model: {model}')
        results[model] = []
        for S in stock_prices_grid:
            results[model].append(LSMC_ML(S, strike, r, sigma, T, M, lamb, model)[0])

    print('  Computing LSMC Poly (1+S+S^2)')
    results['lsmc poly'] = []
    for S in stock_prices_grid:
        results['lsmc poly'].append(LSMC_poly(S, strike, r, sigma, T, M, lamb)[0])

    plt.figure(figsize=(12, 8))
    for i, (name, prices) in enumerate(results.items()):
        plt.plot(stock_prices_grid, prices,
                 color=colors[i % len(colors)], linestyle=styles[i % len(styles)],
                 linewidth=2, marker='o', markersize=4,
                 label=model_labels.get(name, name))
    plt.plot(stock_prices_grid, payoffs, color='black', linestyle=':', linewidth=2,
             label='Payoff function')
    plt.title(f'IS comparison of ML models in LSMC for an American put option\n'
              f'(Strike={strike}, T={T}, sigma={sigma}, r={r}, lambda={lamb})')
    plt.xlabel('Initial stock price $S_0$')
    plt.ylabel('Option price')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'cell1_comparison_lambda{lamb}.png',
                dpi=150, bbox_inches='tight')
    plt.show()


## IS Performance table

In [ ]:
LAMBDAS = [10, 25, 50, 100]
T, M, r, sigma, S0, strike = 1.0, 15000, 0.02, 0.4, 100, 100

MODEL_NAMES = {
    'knn': 'KNN', 'decision_tree': 'Decision Tree',
    'xgboost': 'XGBoost', 'lightgbm': 'LightGBM',
    'logistic_regression': 'Ridge (Linear)', 'random_forest': 'Random Forest',
}

for lamb in LAMBDAS:
    print('\n' + '-' * 50)
    print(f'Lambda = {lamb}')
    print(f'Parameters: S0={S0}, K={strike}, T={T}, r={r}, sigma={sigma}, lambda={lamb}, M={M}')
    print('-' * 50)

    print('Computing LSMC Poly (1+S+S^2) [reference baseline]...', flush=True)
    t0 = time.time()
    val_poly, _ = LSMC_poly(S0, strike, r, sigma, T, M, lamb)
    poly_time = time.time() - t0

    rows = [{'method': 'LSMC Poly (1+S+S²)', 'value': val_poly, 'time': poly_time, 'diff': 0.0}]

    for m in MODEL_NAMES:
        if m == 'xgboost'  and not XGBOOST_AVAILABLE:  continue
        if m == 'lightgbm' and not LIGHTGBM_AVAILABLE: continue
        print(f'Computing {MODEL_NAMES[m]}...', flush=True)
        t0 = time.time()
        val, _, _ = LSMC_ML(S0, strike, r, sigma, T, M, lamb, m)
        ml_time = time.time() - t0
        rows.append({'method': MODEL_NAMES[m], 'value': val, 'time': ml_time,
                     'diff': val - val_poly})

    k2_filename = OUTPUT_DIR / f'single_scenario_summary_lambda{lamb}.csv'
    with open(k2_filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['method', 'value', 'diff vs poly', 'time s'])
        writer.writeheader()
        for row in rows:
            writer.writerow({
                'method': row['method'],
                'value': f"{row['value']:.2f}",
                'diff vs poly': f"{row['diff']:+.2f}",
                'time s': f"{row['time']:.2f}",
            })


## Heatmap of option feature correlations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

T = 1.0
r = 0.02
sigma = 0.4

def poisson(lamb, T):
    poisson_times = np.cumsum(np.random.exponential(1 / lamb, size=int(lamb * T * 2)))
    return poisson_times[poisson_times <= T]

def gbm_paths(S0, r, sigma, T, lamb, M):
    paths = []
    times = []
    for _ in range(M):
        t = poisson(lamb, T)
        if len(t) == 0:
            t = np.array([0, T])
        else:
            t = np.insert(t, 0, 0)
            if t[-1] < T:
                t = np.append(t, T)
        
        B = np.zeros(len(t))
        for i in range(1, len(t)):
            dt = t[i] - t[i-1]
            B[i] = B[i-1] + np.random.normal(0, np.sqrt(dt))
        
        S = S0 * np.exp((r - 0.5 * sigma**2) * t + sigma * B)
        paths.append(S)
        times.append(t)
    return paths, times

def payoff(S, strike):
    return np.maximum(strike - S, 0)

def generate_correlation_dataset(n_samples=5000):
    np.random.seed(42)
    
    data = []
    
    for _ in range(n_samples):
        strike = np.random.uniform(80, 120)
        S0 = np.random.uniform(70, 130)
        
        volume = np.random.uniform(100, 10000)
        
        open_interest = volume * np.random.uniform(0.5, 2.0)
        
        t = np.random.uniform(0, T * 0.9)
        time_to_expiry = T - t
        
        moneyness = S0 / strike
        delta = -np.exp(-r * time_to_expiry) * (1 - np.random.uniform(0, 1) * moneyness)
        
        gamma = np.exp(-r * time_to_expiry) / (S0 * sigma * np.sqrt(2 * np.pi * time_to_expiry)) * \
                np.exp(-0.5 * ((np.log(S0/strike) + (r + 0.5*sigma**2)*time_to_expiry) / \
                (sigma * np.sqrt(time_to_expiry)))**2)
        
        theta = -S0 * sigma / (2 * np.sqrt(time_to_expiry)) * gamma - \
                r * strike * np.exp(-r * time_to_expiry)
        
        vega = S0 * np.sqrt(time_to_expiry) * gamma
        
        implied_volatility = sigma + np.random.normal(0, 0.05)
        
        data.append({
            'strike': strike,
            'volume': volume,
            'open_interest': open_interest,
            'delta': delta,
            'gamma': gamma,
            'theta': theta,
            'vega': vega,
            'implied_volatility': implied_volatility
        })
    
    return pd.DataFrame(data)

df = generate_correlation_dataset(n_samples=5000)


## IS CSV table: LightGBM vs Ridge (LR)

In [ ]:
LAMBDAS = [10, 25, 50, 100]
r, strike, M = 0.02, 100, 15000
S0_values    = [80, 85, 90, 95, 100, 105, 110, 115, 120]
sigma_values = [0.2, 0.4]
T_values     = [1, 2]

ml_models = {'LGBM': 'lightgbm', 'LR': 'logistic_regression'}
if not LIGHTGBM_AVAILABLE:
    print('WARNING: LightGBM not available, using Random Forest as substitute')
    ml_models['LGBM'] = 'random_forest'

fieldnames = ['S0', 'sigma', 'T', 'Ref Price', 'Ref StdErr',
              'Poly Price', 'Poly StdErr', 'Poly Bias']
for short_name in ml_models:
    fieldnames += [f'{short_name} Price', f'{short_name} StdErr', f'{short_name} Bias']

total = len(S0_values) * len(sigma_values) * len(T_values)

for lamb in LAMBDAS:
    filename = OUTPUT_DIR / f'option_prices_lgbm_lr_lambda{lamb}.csv'
    print('\n' + '-' * 50)
    print(f'Lambda = {lamb}    (file: {filename.name})')
    print('-' * 50)
    print(f'Total scenarios: {total}')

    results = []
    scenario_id = 0
    for S0 in S0_values:
        for sigma in sigma_values:
            for T in T_values:
                scenario_id += 1
                print(f'[{scenario_id}/{total}] S0={S0}, sigma={sigma}, T={T}',
                      flush=True)
                row = {'S0': S0, 'sigma': sigma, 'T': T}

                ref_p, ref_e = LSMC_reference_oos(S0, strike, r, sigma, T, 20000, lamb)
                row['Ref Price']  = ref_p
                row['Ref StdErr'] = ref_e

                poly_p, poly_e = LSMC_poly(S0, strike, r, sigma, T, M, lamb)
                row['Poly Price']  = poly_p
                row['Poly StdErr'] = poly_e
                row['Poly Bias']   = poly_p - ref_p

                for short_name, mtype in ml_models.items():
                    p, _, e = LSMC_ML(S0, strike, r, sigma, T, M, lamb, mtype)
                    row[f'{short_name} Price']  = p
                    row[f'{short_name} StdErr'] = e
                    row[f'{short_name} Bias']   = p - ref_p

                results.append(row)

    with open(filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows([
        {k: (f'{v:.2f}' if isinstance(v, float) else v) for k, v in row.items()}
        for row in results
    ])
    print(f'Results saved to: {filename}')

    print('\n' + '-' * 50)
    print(f'IS BIAS SUMMARY (LightGBM + LR + Poly vs Reference), lambda={lamb}')
    print('-' * 50)
    print(f'{"Method":<10} {"Mean Bias":>12} {"RMSE":>10} {"Max |Bias|":>12} {"% < ref":>10}')
    print('-' * 70)
    summary_rows = []
    all_methods = ['Poly'] + list(ml_models.keys())
    for short_name in all_methods:
        biases = np.array([row[f'{short_name} Bias'] for row in results])
        mb = float(biases.mean())
        rmse = float(np.sqrt((biases ** 2).mean()))
        mx = float(np.abs(biases).max())
        pct_below = 100.0 * float((biases < 0).mean())
        print(f'{short_name:<10} {mb:>+12.4f} {rmse:>10.4f} {mx:>12.4f} {pct_below:>9.1f}%')
        summary_rows.append({
            'method': short_name,
            'mean bias': f'{mb:+.2f}',
            'rmse': f'{rmse:.2f}',
            'max abs bias': f'{mx:.2f}',
            'pct below ref': f'{pct_below:.2f}',
        })
    print('-' * 50)


    import collections
    agg = collections.defaultdict(list)
    for row in results:
        agg[row['S0']].append(row)
    S0_sorted = sorted(agg.keys())
    plt.figure(figsize=(11, 6))
    for short_name in all_methods:
        avg_bias = [np.mean([r[f'{short_name} Bias'] for r in agg[s0]]) for s0 in S0_sorted]
        plt.plot(S0_sorted, avg_bias, marker='o', linewidth=2, label=short_name)
    plt.axhline(0, color='black', linestyle='--', alpha=0.6, label='reference')
    plt.xlabel('Initial stock price $S_0$')
    plt.ylabel('Average bias (across sigma and T)')
    plt.title(f'IS bias of methods (LightGBM/LR + Poly) vs reference, lambda={lamb}')
    plt.grid(alpha=0.3)
    plt.legend(ncol=2)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'cell4_bias_lambda{lamb}.png', dpi=150, bbox_inches='tight')
    plt.show()


## IS CSV table: KNN / DT / XGBoost / RF

In [ ]:
LAMBDAS = [10, 25, 50, 100]
r, strike, M = 0.02, 100, 15000
S0_values    = [80, 85, 90, 95, 100, 105, 110, 115, 120]
sigma_values = [0.2, 0.4]
T_values     = [1, 2]

ml_models = {'KNN': 'knn', 'DT': 'decision_tree',
             'XGB': 'xgboost', 'RF': 'random_forest'}
if not XGBOOST_AVAILABLE:
    print('WARNING: XGBoost unavailable, skipping')
    del ml_models['XGB']

fieldnames = ['S0', 'sigma', 'T', 'Ref Price', 'Ref StdErr',
              'Poly Price', 'Poly StdErr', 'Poly Bias']
for short_name in ml_models:
    fieldnames += [f'{short_name} Price', f'{short_name} StdErr', f'{short_name} Bias']

total = len(S0_values) * len(sigma_values) * len(T_values)

for lamb in LAMBDAS:
    filename = OUTPUT_DIR / f'option_prices_ml_comparison_lambda{lamb}.csv'
    print('\n' + '-' * 50)
    print(f'Lambda = {lamb}    (file: {filename.name})')
    print('-' * 50)
    print(f'Total scenarios: {total}')

    results = []
    scenario_id = 0
    for S0 in S0_values:
        for sigma in sigma_values:
            for T in T_values:
                scenario_id += 1
                print(f'[{scenario_id}/{total}] S0={S0}, sigma={sigma}, T={T}',
                      flush=True)
                row = {'S0': S0, 'sigma': sigma, 'T': T}

                ref_p, ref_e = LSMC_reference_oos(S0, strike, r, sigma, T, 20000, lamb)
                row['Ref Price']  = ref_p
                row['Ref StdErr'] = ref_e

                poly_p, poly_e = LSMC_poly(S0, strike, r, sigma, T, M, lamb)
                row['Poly Price']  = poly_p
                row['Poly StdErr'] = poly_e
                row['Poly Bias']   = poly_p - ref_p

                for short_name, mtype in ml_models.items():
                    p, _, e = LSMC_ML(S0, strike, r, sigma, T, M, lamb, mtype)
                    row[f'{short_name} Price']  = p
                    row[f'{short_name} StdErr'] = e
                    row[f'{short_name} Bias']   = p - ref_p

                results.append(row)

    with open(filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows([
        {k: (f'{v:.2f}' if isinstance(v, float) else v) for k, v in row.items()}
        for row in results
    ])
    print(f'Results saved to: {filename}')

    import collections
    agg = collections.defaultdict(list)
    for row in results:
        agg[row['S0']].append(row)
    S0_sorted = sorted(agg.keys())
    plt.figure(figsize=(11, 6))
    for short_name in all_methods:
        avg_bias = [np.mean([r[f'{short_name} Bias'] for r in agg[s0]]) for s0 in S0_sorted]
        plt.plot(S0_sorted, avg_bias, marker='o', linewidth=2, label=short_name)
    plt.axhline(0, color='black', linestyle='--', alpha=0.6, label='reference')
    plt.xlabel('Initial stock price $S_0$')
    plt.ylabel('Average bias (across sigma and T)')
    plt.title(f'IS bias of methods (KNN/DT/XGB/RF + Poly) vs reference, lambda={lamb}')
    plt.grid(alpha=0.3)
    plt.legend(ncol=2)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'cell6_bias_lambda{lamb}.png', dpi=150, bbox_inches='tight')
    plt.show()


## IS Validation: reference price and bias of each method


In [ ]:
def LSMC_reference(S0, strike, r, sigma, T, M, lamb):
    all_paths, all_time_grids = gbm_paths(S0, r, sigma, T, lamb, M)
    cash_flows = [float(payoff(p[-1], strike)) for p in all_paths]
    cash_times = [float(t_grid[-1]) for t_grid in all_time_grids]
    max_steps = max(len(p) for p in all_paths)
    for step in range(max_steps - 2, 0, -1):
        path_indices, X_list, Y_list = [], [], []
        for idx, (path, t_grid) in enumerate(zip(all_paths, all_time_grids)):
            if step >= len(path):
                continue
            current_price = path[step]
            if current_price >= strike:
                continue
            current_time = float(t_grid[step])
            path_indices.append(idx)
            X_list.append(current_price)
            Y_list.append(cash_flows[idx] * np.exp(-r * (cash_times[idx] - current_time)))
        if len(path_indices) < 4:
            continue
        X = np.array(X_list); Y = np.array(Y_list)
        A = np.vstack([np.ones_like(X), X, X**2, X**3]).T
        try:
            coeff = np.linalg.lstsq(A, Y, rcond=None)[0]
            continuation = A @ coeff
        except Exception:
            continue
        for i, idx in enumerate(path_indices):
            ex_val = float(payoff(all_paths[idx][step], strike))
            if ex_val > continuation[i]:
                cash_flows[idx] = ex_val
                cash_times[idx] = float(all_time_grids[idx][step])
    discounted = np.array([cf * np.exp(-r * ct) for cf, ct in zip(cash_flows, cash_times)])
    continuation_at_zero = float(discounted.mean())
    ex_val_at_zero = float(payoff(S0, strike))
    price = ex_val_at_zero if ex_val_at_zero > continuation_at_zero else continuation_at_zero
    return price, float(discounted.std() / np.sqrt(len(discounted)))


LAMBDAS = [10, 25, 50, 100]
T, r, sigma, strike = 1.0, 0.05, 0.2, 90
M_ref  = 20000
M_test = 20000
S0_grid = np.array([70, 80, 85, 90, 95, 100, 110, 120, 130])

methods = ['knn', 'decision_tree', 'random_forest', 'logistic_regression']
if XGBOOST_AVAILABLE:  methods.insert(2, 'xgboost')
if LIGHTGBM_AVAILABLE: methods.insert(3 if XGBOOST_AVAILABLE else 2, 'lightgbm')

ref_data = {}

for lamb in LAMBDAS:
    print('\n' + '-' * 60)
    print(f'Lambda = {lamb}')
    print('-' * 60)
    print(f'Computing reference price (LSMC cubic, M={M_ref})...')
    ref_prices, ref_errs = [], []
    for s0 in S0_grid:
        p, e = LSMC_reference(s0, strike, r, sigma, T, M_ref, lamb)
        ref_prices.append(p)
        ref_errs.append(e)
    ref_data[lamb] = {'prices': list(ref_prices), 'errs': list(ref_errs)}

    prices_per_method = {m: [] for m in methods}
    prices_per_method['lsmc_poly'] = []

    print(f'\nComputing tested IS methods (M_test={M_test})...')
    for s0 in S0_grid:
        for m in methods:
            prices_per_method[m].append(LSMC_ML(s0, strike, r, sigma, T, M_test, lamb, m)[0])
        prices_per_method['lsmc_poly'].append(LSMC_poly(s0, strike, r, sigma, T, M_test, lamb)[0])

    print(f'\n{"Method":<22} {"Mean Bias":>12} {"RMSE":>10} {"Max |Bias|":>12}')
    print('-' * 60)
    ref_arr = np.array(ref_prices)
    for m, prices in prices_per_method.items():
        arr = np.array(prices)
        mb = float((arr - ref_arr).mean())
        rmse = float(np.sqrt(((arr - ref_arr) ** 2).mean()))
        max_abs = float(np.abs(arr - ref_arr).max())
        print(f'{m:<22} {mb:>+12.4f} {rmse:>10.4f} {max_abs:>12.4f}')

    plt.figure(figsize=(12, 6))
    for m, prices in prices_per_method.items():
        bias = np.array(prices) - ref_arr
        plt.plot(S0_grid, bias, marker='o', linewidth=2, label=m)
    plt.axhline(0, color='black', linestyle='--', alpha=0.5, label='reference')
    plt.xlabel('Initial stock price $S_0$')
    plt.ylabel('Bias = method price - reference price')
    plt.title(f'IS bias of each method vs reference LSMC (cubic, M_ref={M_ref})\n'
              f'(K={strike}, T={T}, sigma={sigma}, r={r}, lambda={lamb})')
    plt.grid(alpha=0.3)
    plt.legend(ncol=2)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'cell6_bias_vs_S0_lambda{lamb}.png',
                dpi=150, bbox_inches='tight')
    plt.show()




## OOS Model comparison: plot and diagnostics

In [ ]:
LAMBDAS = [10, 25, 50, 100]
T, M, r, sigma, strike = 1.0, 10000, 0.05, 0.2, 90
test_s0 = 100

stock_prices_grid = np.linspace(70, 130, 30)

available_models = ['knn', 'decision_tree', 'random_forest']
if XGBOOST_AVAILABLE:
    available_models.append('xgboost')
if LIGHTGBM_AVAILABLE:
    available_models.append('lightgbm')

payoffs = [payoff(S, strike) for S in stock_prices_grid]
model_labels = {m: m.upper() for m in available_models}

colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'teal']
styles = ['-', '--', '-.', ':', '-', '--', '-.']

for lamb in LAMBDAS:
    print('\n' + '-' * 40)
    print(f'Lambda = {lamb}')
    print('-' * 40)

    results = {}
    for model in available_models:
        print(f'  Computing OOS for model: {model}')
        results[model] = []
        for S in stock_prices_grid:
            results[model].append(LSMC_ML_oos(S, strike, r, sigma, T, M, lamb, model)[0])

    plt.figure(figsize=(12, 8))
    for i, (name, prices) in enumerate(results.items()):
        plt.plot(stock_prices_grid, prices,
                 color=colors[i % len(colors)], linestyle=styles[i % len(styles)],
                 linewidth=2, marker='o', markersize=4,
                 label=model_labels.get(name, name))
    plt.plot(stock_prices_grid, payoffs, color='black', linestyle=':', linewidth=2,
             label='Payoff function')
    plt.title(f'OOS comparison of high capacity ML models in LSMC for an American put option\n'
              f'(Strike={strike}, T={T}, sigma={sigma}, r={r}, lambda={lamb})')
    plt.xlabel('Initial stock price $S_0$')
    plt.ylabel('Option price')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'oos_comparison_lambda{lamb}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

## OOS Performance table

In [ ]:
LAMBDAS = [10, 25, 50, 100]
T, M, r, sigma, S0, strike = 1.0, 15000, 0.02, 0.4, 100, 100

MODEL_NAMES = {
    'knn': 'KNN', 'decision_tree': 'Decision Tree',
    'xgboost': 'XGBoost', 'lightgbm': 'LightGBM',
    'random_forest': 'Random Forest',
}

for lamb in LAMBDAS:
    print('\n' + '-' * 50)
    print(f'Lambda = {lamb}')
    print(f'Parameters: S0={S0}, K={strike}, T={T}, r={r}, sigma={sigma}, lambda={lamb}, M={M}')
    print('-' * 50)

    rows = []
    for m in MODEL_NAMES:
        if m == 'xgboost'  and not XGBOOST_AVAILABLE:  continue
        if m == 'lightgbm' and not LIGHTGBM_AVAILABLE: continue
        print(f'Computing {MODEL_NAMES[m]} OOS...', flush=True)
        t0 = time.time()
        val, _, _ = LSMC_ML_oos(S0, strike, r, sigma, T, M, lamb, m)
        ml_time = time.time() - t0
        rows.append({'method': MODEL_NAMES[m], 'value': val, 'time': ml_time})

    k8_filename = OUTPUT_DIR / f'oos_single_scenario_summary_lambda{lamb}.csv'
    with open(k8_filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['method', 'value', 'time s'])
        writer.writeheader()
        for row in rows:
            writer.writerow({
                'method': row['method'],
                'value': f"{row['value']:.2f}",
                'time s': f"{row['time']:.2f}",
            })


## OOS CSV table: LightGBM / XGBoost vs Reference

In [ ]:
LAMBDAS = [10, 25, 50, 100]
r_loc, strike_loc, M_loc = 0.02, 100, 15000
S0_values    = [80, 85, 90, 95, 100, 105, 110, 115, 120]
sigma_values = [0.2, 0.4]
T_values     = [1, 2]

ml_models = {}
if LIGHTGBM_AVAILABLE:
    ml_models['LGBM'] = 'lightgbm'
if XGBOOST_AVAILABLE:
    ml_models['XGB'] = 'xgboost'
if not ml_models:
    print('WARNING: neither LightGBM nor XGBoost available — falling back to Random Forest')
    ml_models['RF'] = 'random_forest'

fieldnames = ['S0', 'sigma', 'T', 'Ref Price', 'Ref StdErr']
for short_name in ml_models:
    fieldnames += [f'{short_name} Price', f'{short_name} StdErr', f'{short_name} Bias']

total_scenarios = len(S0_values) * len(sigma_values) * len(T_values)

for lamb_loc in LAMBDAS:
    filename = OUTPUT_DIR / f'option_prices_oos_lgbm_xgb_lambda{lamb_loc}.csv'
    print('\n' + '-' * 50)
    print(f'Lambda = {lamb_loc}    (file: {filename.name})')
    print('-' * 50)
    print(f'Number of scenarios: {total_scenarios}')
    print(f'M_test_oos={M_loc}, M_ref={M_loc}, lambda={lamb_loc}, r={r_loc}, K={strike_loc}')

    results_rows = []
    scenario_id = 0
    for S0 in S0_values:
        for sigma_loc in sigma_values:
            for T_loc in T_values:
                scenario_id += 1
                row = {'S0': S0, 'sigma': sigma_loc, 'T': T_loc}

                print(f'[{scenario_id}/{total_scenarios}] S0={S0}, sigma={sigma_loc}, T={T_loc}',
                      flush=True)

                ref_p, ref_e = LSMC_reference_oos(S0, strike_loc, r_loc, sigma_loc,
                                              T_loc, M_loc, lamb_loc)
                row['Ref Price']  = ref_p
                row['Ref StdErr'] = ref_e

                for short_name, mtype in ml_models.items():
                    p, _, e = LSMC_ML_oos(S0, strike_loc, r_loc, sigma_loc,
                                          T_loc, M_loc, lamb_loc, mtype)
                    row[f'{short_name} Price']  = p
                    row[f'{short_name} StdErr'] = e
                    row[f'{short_name} Bias']   = p - ref_p

                results_rows.append(row)

    with open(filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows([
        {k: (f'{v:.2f}' if isinstance(v, float) else v) for k, v in row.items()}
        for row in results_rows
    ])
    print(f'Results saved to: {filename}')

    import collections
    agg = collections.defaultdict(list)
    for row in results_rows:
        agg[row['S0']].append(row)
    S0_sorted = sorted(agg.keys())
    plt.figure(figsize=(11, 6))
    for short_name in ml_models:
        avg_bias = [np.mean([r[f'{short_name} Bias'] for r in agg[s0]]) for s0 in S0_sorted]
        plt.plot(S0_sorted, avg_bias, marker='o', linewidth=2, label=short_name)
    plt.axhline(0, color='black', linestyle='--', alpha=0.6, label='reference')
    plt.xlabel('Initial stock price $S_0$')
    plt.ylabel('Average bias (across sigma and T)')
    plt.title(f'OOS bias of gradient boosting methods (LightGBM, XGBoost) '
              f'vs reference, lambda={lamb_loc}')
    plt.grid(alpha=0.3)
    plt.legend(ncol=2)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'cell9_avg_bias_lambda{lamb_loc}.png',
                dpi=150, bbox_inches='tight')
    plt.show()


## OOS CSV table: KNN / Decision Tree / Random Forest vs Reference

In [ ]:
LAMBDAS = [10, 25, 50, 100]
r_loc, strike_loc, M_loc = 0.02, 100, 15000
S0_values    = [80, 85, 90, 95, 100, 105, 110, 115, 120]
sigma_values = [0.2, 0.4]
T_values     = [1, 2]

ml_models = {'KNN': 'knn', 'DT': 'decision_tree', 'RF': 'random_forest'}

fieldnames = ['S0', 'sigma', 'T', 'Ref Price', 'Ref StdErr']
for short_name in ml_models:
    fieldnames += [f'{short_name} Price', f'{short_name} StdErr', f'{short_name} Bias']

total_scenarios = len(S0_values) * len(sigma_values) * len(T_values)

for lamb_loc in LAMBDAS:
    filename = OUTPUT_DIR / f'option_prices_oos_knn_dt_rf_lambda{lamb_loc}.csv'
    print('\n' + '-' * 50)
    print(f'Lambda = {lamb_loc}    (file: {filename.name})')
    print('-' * 50)
    print(f'Number of scenarios: {total_scenarios}')
    print(f'M_test_oos={M_loc}, M_ref={M_loc}, lambda={lamb_loc}, r={r_loc}, K={strike_loc}')

    results_rows = []
    scenario_id = 0
    for S0 in S0_values:
        for sigma_loc in sigma_values:
            for T_loc in T_values:
                scenario_id += 1
                row = {'S0': S0, 'sigma': sigma_loc, 'T': T_loc}

                print(f'[{scenario_id}/{total_scenarios}] S0={S0}, sigma={sigma_loc}, T={T_loc}',
                      flush=True)

                ref_p, ref_e = LSMC_reference_oos(S0, strike_loc, r_loc, sigma_loc,
                                              T_loc, M_loc, lamb_loc)
                row['Ref Price']  = ref_p
                row['Ref StdErr'] = ref_e

                for short_name, mtype in ml_models.items():
                    p, _, e = LSMC_ML_oos(S0, strike_loc, r_loc, sigma_loc,
                                          T_loc, M_loc, lamb_loc, mtype)
                    row[f'{short_name} Price']  = p
                    row[f'{short_name} StdErr'] = e
                    row[f'{short_name} Bias']   = p - ref_p

                results_rows.append(row)

    with open(filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows([
        {k: (f'{v:.2f}' if isinstance(v, float) else v) for k, v in row.items()}
        for row in results_rows
    ])
    print(f'Results saved to: {filename}')


    import collections
    agg = collections.defaultdict(list)
    for row in results_rows:
        agg[row['S0']].append(row)
    S0_sorted = sorted(agg.keys())
    plt.figure(figsize=(11, 6))
    for short_name in ml_models:
        avg_bias = [np.mean([r[f'{short_name} Bias'] for r in agg[s0]]) for s0 in S0_sorted]
        plt.plot(S0_sorted, avg_bias, marker='o', linewidth=2, label=short_name)
    plt.axhline(0, color='black', linestyle='--', alpha=0.6, label='reference')
    plt.xlabel('Initial stock price $S_0$')
    plt.ylabel('Average bias (across sigma and T)')
    plt.title(f'OOS bias of remaining ML methods (KNN, DT, RF) '
              f'vs reference, lambda={lamb_loc}')
    plt.grid(alpha=0.3)
    plt.legend(ncol=2)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'cell10_avg_bias_lambda{lamb_loc}.png',
                dpi=150, bbox_inches='tight')
    plt.show()


## OOS Validation: reference price and bias

In [ ]:
LAMBDAS = [10, 25, 50, 100]
T, r, sigma, strike = 1.0, 0.05, 0.2, 90
M_ref  = 20000
M_test = 20000
S0_grid = np.array([70, 80, 85, 90, 95, 100, 110, 120, 130])

methods = ['knn', 'decision_tree', 'random_forest', 'logistic_regression']
if XGBOOST_AVAILABLE:  methods.insert(2, 'xgboost')
if LIGHTGBM_AVAILABLE: methods.insert(3 if XGBOOST_AVAILABLE else 2, 'lightgbm')

try:
    _ref_cache = ref_data  # populated by previously generated reference
except NameError:
    _ref_cache = {}

for lamb in LAMBDAS:
    print('\n' + '-' * 60)
    print(f'Lambda = {lamb}')
    print('-' * 60)

    if lamb in _ref_cache:
        ref_prices = _ref_cache[lamb]['prices']
        ref_errs   = _ref_cache[lamb]['errs']
        print(f'Using cached reference prices for lambda={lamb}.')
    else:
        print(f'Cached reference not found; recomputing (LSMC cubic, M={M_ref})...')
        ref_prices, ref_errs = [], []
        for s0 in S0_grid:
            p, e = LSMC_reference(s0, strike, r, sigma, T, M_ref, lamb)
            ref_prices.append(p)
            ref_errs.append(e)

    prices_per_method = {m: [] for m in methods}
    prices_per_method['lsmc_poly_oos'] = []

    print(f'\nComputing tested OOS methods (M_test={M_test})...')
    for s0 in S0_grid:
        for m in methods:
            prices_per_method[m].append(LSMC_ML_oos(s0, strike, r, sigma, T, M_test, lamb, m)[0])
        prices_per_method['lsmc_poly_oos'].append(
            LSMC_ML_oos(s0, strike, r, sigma, T, M_test, lamb, 'ridge')[0])

    print(f'\n{"Method":<22} {"Mean Bias":>12} {"RMSE":>10} {"Max |Bias|":>12}')
    print('-' * 60)
    ref_arr = np.array(ref_prices)
    for m, prices in prices_per_method.items():
        arr = np.array(prices)
        mb = float((arr - ref_arr).mean())
        rmse = float(np.sqrt(((arr - ref_arr) ** 2).mean()))
        max_abs = float(np.abs(arr - ref_arr).max())
        print(f'{m:<22} {mb:>+12.4f} {rmse:>10.4f} {max_abs:>12.4f}')

    plt.figure(figsize=(12, 6))
    for m, prices in prices_per_method.items():
        bias = np.array(prices) - ref_arr
        plt.plot(S0_grid, bias, marker='o', linewidth=2, label=m)
    plt.axhline(0, color='black', linestyle='--', alpha=0.5, label='reference')
    plt.xlabel('Initial stock price $S_0$')
    plt.ylabel('Bias = method price - reference price')
    plt.title(f'OOS bias of each method vs reference LSMC (cubic, M_ref={M_ref})\n'
              f'(K={strike}, T={T}, sigma={sigma}, r={r}, lambda={lamb})')
    plt.grid(alpha=0.3)
    plt.legend(ncol=2)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'oos_bias_vs_S0_lambda{lamb}.png',
                dpi=150, bbox_inches='tight')
    plt.show()


